# Optiver 方案一：通用基线（线性/逻辑回归）

## 简介通用基线方案：数值特征标准化、类别特征独热编码，使用 Ridge（回归）或 LogisticRegression（分类）进行 5 折交叉验证与全量拟合。

In [ ]:
import osimport numpy as npimport pandas as pdfrom sklearn.compose import ColumnTransformerfrom sklearn.preprocessing import OneHotEncoder, StandardScalerfrom sklearn.pipeline import Pipelinefrom sklearn.model_selection import KFold, StratifiedKFoldfrom sklearn.metrics import mean_squared_error, r2_score, roc_auc_score, f1_scorefrom sklearn.linear_model import Ridge, LogisticRegressionnp.random.seed(42)

## 数据加载

In [ ]:
DATA_DIRS = ["./data", "."]TRAIN_FILES = ["train.csv", "optiver_train.csv"]TEST_FILES = ["test.csv", "optiver_test.csv"]def find_dataset_file(names):    for d in DATA_DIRS:        for n in names:            p = os.path.join(d, n)            if os.path.exists(p):                return p    return Nonetrain_path = find_dataset_file(TRAIN_FILES)test_path = find_dataset_file(TEST_FILES)train_df = pd.read_csv(train_path) if train_path else Nonetest_df = pd.read_csv(test_path) if test_path else Noneprint("train_path", train_path)print("test_path", test_path)print(train_df.shape if train_df is not None else None)print(test_df.shape if test_df is not None else None)

## 目标列与任务类型识别

In [ ]:
def detect_target(df):    cols = df.columns.tolist()    candidates = []    for c in cols:        cl = c.lower()        if ("target" in cl) or ("movement" in cl) or (cl == "label") or (cl == "y"):            candidates.append(c)    if candidates:        return candidates[0]    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()    for c in numeric_cols[::-1]:        if df[c].nunique() < len(df):            return c    return cols[-1]def detect_task_type(y):    uniq = pd.unique(y)    if pd.api.types.is_numeric_dtype(y):        ratio = len(uniq) / max(1, len(y))        if len(uniq) <= 20 and ratio < 0.02:            return "classification"        return "regression"    return "classification"target_col = detect_target(train_df) if train_df is not None else Nonetask_type = detect_task_type(train_df[target_col]) if train_df is not None else Noneprint("target_col", target_col)print("task_type", task_type)

## 预处理管线

In [ ]:
def build_preprocessor(df, target):    X = df.drop(columns=[target])    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()    cat_cols = [c for c in X.columns.tolist() if c not in num_cols]    transformers = []    if num_cols:        transformers.append(("num", StandardScaler(), num_cols))    if cat_cols:        transformers.append(("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols))    return ColumnTransformer(transformers=transformers)

## 交叉验证评估

In [ ]:
def run_cv(df, target, task):    X = df.drop(columns=[target])    y = df[target]    if task == "regression":        model = Ridge(alpha=1.0)        cv = KFold(n_splits=5, shuffle=True, random_state=42)        pipe = Pipeline(steps=[("prep", build_preprocessor(df, target)), ("model", model)])        scores = []        for tr, va in cv.split(X):            Xtr, Xva = X.iloc[tr], X.iloc[va]            ytr, yva = y.iloc[tr], y.iloc[va]            pipe.fit(Xtr, ytr)            p = pipe.predict(Xva)            mse = mean_squared_error(yva, p)            r2 = r2_score(yva, p)            scores.append((mse, r2))        return scores, pipe    else:        model = LogisticRegression(max_iter=1000)        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)        pipe = Pipeline(steps=[("prep", build_preprocessor(df, target)), ("model", model)])        scores = []        for tr, va in cv.split(X, y):            Xtr, Xva = X.iloc[tr], X.iloc[va]            ytr, yva = y.iloc[tr], y.iloc[va]            pipe.fit(Xtr, ytr)            proba = pipe.predict_proba(Xva)            if proba.shape[1] == 2:                auc = roc_auc_score(yva, proba[:, 1])                scores.append(auc)            else:                pred = pipe.predict(Xva)                f1 = f1_score(yva, pred, average="macro")                scores.append(f1)        return scores, pipescores, pipe = run_cv(train_df, target_col, task_type) if train_df is not None else (None, None)print("cv_scores", None if scores is None else scores[:3])

## 全量拟合与推断

In [ ]:
def fit_full_and_predict(train_df, test_df, target, task):    model = Ridge(alpha=1.0) if task == "regression" else LogisticRegression(max_iter=1000)    pipe = Pipeline(steps=[("prep", build_preprocessor(train_df, target)), ("model", model)])    pipe.fit(train_df.drop(columns=[target]), train_df[target])    if test_df is None:        return None, pipe    pred = pipe.predict(test_df)    return pred, pipepred, pipe = fit_full_and_predict(train_df, test_df, target_col, task_type) if train_df is not None else (None, None)if pred is not None:    id_col = None    if test_df is not None:        for c in ["row_id", "id"]:            if c in test_df.columns:                id_col = c                break    sub = pd.DataFrame({id_col if id_col else "id": test_df[id_col] if id_col else np.arange(len(pred)), "prediction": pred})    sub_path = "submission_scheme1.csv"    sub.to_csv(sub_path, index=False)    print(sub_path)else:    print(None)